In [ ]:
%pip install pandas
%pip install matplotlib

In [ ]:
import pandas as pd
import json
import pandas as pd
from datetime import datetime

def stream_jsonl(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            yield json.loads(line)

df = pd.read_json('data/processed_data/results.jsonl', orient='records', lines=True)
df.index = df['index']
df.drop(columns=['index'], inplace=True)



df['timestamp'] = None
df['model'] = None
df['year'] = None
df['language'] = None


for i, row in enumerate(stream_jsonl('data/wildchat/filtered_15k.jsonl')):
    if i in df.index:
        df.loc[i, 'timestamp'] = row['timestamp']
        df.loc[i, 'model'] = row['model']
        # Extract year from timestamp string
        if row['timestamp']:
            timestamp_dt = pd.to_datetime(row['timestamp'])
        df.loc[i, 'year'] = timestamp_dt.year
        df.loc[i, 'language'] = row['language']

df.head()

In [ ]:
df.columns

In [ ]:
import math

def calculate_turns(conversation):
    # skip the first message
    return math.ceil((len(conversation[1:])) / 2)

def extract_follow_up_data(follow_up_data_row):
    suggestions_given_per_turn = []
    suggestion_used_flags_per_turn = []
    suggestion_used_counts_per_turn = []
    strict_suggestion_used_flags_per_turn = []
    strict_suggestion_used_counts_per_turn = []

    for turn in follow_up_data_row['conversation_turns']:
        # print(turn)
        ai_message_info = turn['AIMessageInformation']
        user_message_info = turn['UserMessageInformation']
        suggestions_offered = ai_message_info['follow_up_suggestions']['follow_up_suggestions']

        turn_suggestion_ids = [suggestion['suggestion_id'] for suggestion in suggestions_offered]
        used_suggestion_ids = user_message_info['used_follow_up_suggestions_ids']
        strict_used_suggestion_ids = list(set(turn_suggestion_ids) & set(used_suggestion_ids))

        num_suggestions_offered = len(suggestions_offered)
        
        
        suggestions_given_per_turn.append(num_suggestions_offered)
        strict_suggestion_used_counts_per_turn.append(len(strict_used_suggestion_ids))

        suggestion_was_used = len(used_suggestion_ids) > 0
        strict_suggestion_was_used = len(strict_used_suggestion_ids) > 0
        
        strict_suggestion_used_flags_per_turn.append(strict_suggestion_was_used)
        
        suggestion_used_flags_per_turn.append(suggestion_was_used)
        suggestion_used_counts_per_turn.append(len(used_suggestion_ids))
        # turn_suggestion_ids = [suggestion['suggestion_id'] for suggestion in suggestions_offered]
        # used_suggestion_ids = turn['UserMessageInformation']['used_follow_up_suggestions_ids']
        # strict_used_suggestion_ids = list(set(turn_suggestion_ids) & set(used_suggestion_ids))
    
    # Calculate total after the loop completes
    total_suggestions_given_count = sum(suggestions_given_per_turn)
    
    return {
        'suggestions_given_per_turn': suggestions_given_per_turn,
        'suggestion_used_flags_per_turn': suggestion_used_flags_per_turn,
        'suggestion_used_counts_per_turn': suggestion_used_counts_per_turn,
        'num_turns_with_suggestions_used': sum(suggestion_used_flags_per_turn),
        'total_suggestions_used': sum(suggestion_used_counts_per_turn),
        # 'total_suggestions_offered': sum(suggestions_given_per_turn),
        'strict_suggestion_used_counts_per_turn': strict_suggestion_used_counts_per_turn,
        'strict_suggestion_used_flags_per_turn': strict_suggestion_used_flags_per_turn,
        'total_suggestions_given_count': total_suggestions_given_count
    }

# Initialize columns with object dtype to store lists
# Use .loc to avoid SettingWithCopyWarning
df.loc[:, 'suggestions_given_per_turn'] = None
df.loc[:, 'suggestion_used_flags_per_turn'] = None
df.loc[:, 'suggestion_used_counts_per_turn'] = None
df.loc[:, 'strict_suggestion_used_counts_per_turn'] = None
df.loc[:, 'strict_suggestion_used_flags_per_turn'] = None
df.loc[:, 'total_suggestions_given_count'] = None

df.loc[:, 'suggestions_given_per_turn'] = df.loc[:, 'suggestions_given_per_turn'].astype('object')
df.loc[:, 'suggestion_used_flags_per_turn'] = df.loc[:, 'suggestion_used_flags_per_turn'].astype('object')
df.loc[:, 'suggestion_used_counts_per_turn'] = df.loc[:, 'suggestion_used_counts_per_turn'].astype('object')
df.loc[:, 'strict_suggestion_used_counts_per_turn'] = df.loc[:, 'strict_suggestion_used_counts_per_turn'].astype('object')
df.loc[:, 'strict_suggestion_used_flags_per_turn'] = df.loc[:, 'strict_suggestion_used_flags_per_turn'].astype('object')
df.loc[:, 'total_suggestions_used'] = None
df.loc[:, 'turns_match'] = None


# apply the function to the follow_up_data column and create 5 new columns with the results
for i, row in df.iterrows():
    extracted_data = extract_follow_up_data(row['follow_up_data'])
    df.at[i, 'suggestions_given_per_turn'] = extracted_data['suggestions_given_per_turn']
    df.at[i, 'suggestion_used_flags_per_turn'] = extracted_data['suggestion_used_flags_per_turn']
    df.at[i, 'suggestion_used_counts_per_turn'] = extracted_data['suggestion_used_counts_per_turn']
    df.loc[i, 'num_turns_with_suggestions_used'] = extracted_data['num_turns_with_suggestions_used']
    df.loc[i, 'total_suggestions_used'] = extracted_data['total_suggestions_used']
    df.at[i, 'total_suggestions_given_count'] = extracted_data['total_suggestions_given_count']
    df.at[i, 'strict_suggestion_used_counts_per_turn'] = extracted_data['strict_suggestion_used_counts_per_turn']
    df.at[i, 'strict_suggestion_used_flags_per_turn'] = extracted_data['strict_suggestion_used_flags_per_turn']
    df.at[i, 'total_suggestions_given_count'] = extracted_data['total_suggestions_given_count']
df['turns'] = df.apply(lambda row: calculate_turns(row['conversation']), axis=1)
df.loc[:, 'turns_match'] = df.apply(lambda row: len(row['suggestions_given_per_turn']) == row.turns, axis=1)
df = df[df['turns_match']].copy()

In [ ]:
df

In [ ]:
flatteded_data = []
for i, row in df.iterrows():
    turns = row.turns
    conversation_hash = row.conversation_hash
    year = row.year
    model = row.model
    timestamp = row.timestamp
    
    
    for turn in range(turns):
        
        sugestion_given_count = row['suggestions_given_per_turn'][turn]
        suggestion_used_count = row['suggestion_used_counts_per_turn'][turn]
        suggestion_used_flag = row['suggestion_used_flags_per_turn'][turn]

        strict_suggestion_used_count = row['strict_suggestion_used_counts_per_turn'][turn]
        strict_suggestion_used_flag = row['strict_suggestion_used_flags_per_turn'][turn]
        conversation_continued = turn < turns - 1
        
        flatteded_data.append({
            'conversation_hash': conversation_hash,
            'year': year,
            'model': model,
            'timestamp': timestamp,
            'turn': turn + 1,
            'turns': turns,
            'sugestion_given_count': sugestion_given_count,
            'suggestion_used_count': suggestion_used_count,
            'suggestion_used_flag': suggestion_used_flag,
            'strict_suggestion_used_count': strict_suggestion_used_count,
            'strict_suggestion_used_flag': strict_suggestion_used_flag,
            'conversation_continued': conversation_continued
        })

minified_df = pd.DataFrame(flatteded_data)
minified_df.to_json('data/processed_data/minified_results_flattened.jsonl', orient='records', lines=True)
        

In [ ]:
df.columns

In [ ]:
df[['conversation_hash', 'timestamp', 'model', 'year',
       'suggestions_given_per_turn', 'suggestion_used_flags_per_turn',
       'suggestion_used_counts_per_turn',
       'strict_suggestion_used_counts_per_turn',
       'strict_suggestion_used_flags_per_turn',
       'total_suggestions_given_count', 'total_suggestions_used', 'num_turns_with_suggestions_used',
       'turns']].to_json('data/processed_data/minified_results.jsonl', orient='records', lines=True)

In [ ]:
from matplotlib import pyplot as plt
# graph the suggestion count per year
df.groupby('year')['total_suggestions_given_count'].mean().plot(kind='bar')
plt.show()


In [ ]:
# graph the suggestion count per year where suggestions are used excluding 0
df[df['total_suggestions_given_count'] > 0].groupby('year')['total_suggestions_given_count'].mean().plot(kind='bar')
plt.show()

In [ ]:
df.year.value_counts()